# Chapitre 4 - Exercices 4.1 à 4.5

- **4.1** : problème aux valeurs propres généralisé ;
- **4.2** : LDA et moindres carrés à deux classes ;
- **4.3** : LDA sur les réponses ajustées de la régression sur indicatrices ;
- **4.4** : Newton-Raphson pour la logistique multinomiale ;
- **4.5** : régression logistique sur des données séparables.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch04_classification as c4
set_style()

## Exercice 4.1 : Problème aux valeurs propres généralisé

**Énoncé.** Montrer comment résoudre le problème
$\max_a a^T\mathbf{B}a$ sous la contrainte $a^T\mathbf{W}a = 1$ en le ramenant à un
problème aux valeurs propres standard. (Ici $\mathbf{B}$ est symétrique semi-définie
positive et $\mathbf{W}$ symétrique définie positive, comme les covariances inter- et
intra-classes de la LDA de rang réduit.)

### Rappels

- **Théorème spectral.** Une matrice symétrique réelle $\mathbf{S}$ s'écrit
  $\mathbf{S} = \mathbf{U}\mathbf{D}\mathbf{U}^T$ avec $\mathbf{U}$ orthogonale (colonnes
  $u_j$ orthonormées) et $\mathbf{D} = \operatorname{diag}(\lambda_1 \ge \dots \ge \lambda_p)$.
- **Racine carrée.** Si $\mathbf{W} = \mathbf{U}\mathbf{D}\mathbf{U}^T$ est définie positive
  ($\lambda_j > 0$), on pose $\mathbf{W}^{1/2} = \mathbf{U}\mathbf{D}^{1/2}\mathbf{U}^T$ :
  elle est symétrique, inversible, d'inverse $\mathbf{W}^{-1/2} = \mathbf{U}\mathbf{D}^{-1/2}\mathbf{U}^T$,
  et $\mathbf{W}^{1/2}\mathbf{W}^{1/2} = \mathbf{W}$.

### Lemme : maximum d'une forme quadratique sur la sphère

Soit $\mathbf{S} = \mathbf{U}\mathbf{D}\mathbf{U}^T$ symétrique. Alors
$\max_{\|b\| = 1} b^T\mathbf{S}b = \lambda_1$, atteint en $b = u_1$.

*Preuve.* Écrivons $b = \sum_j c_ju_j$ dans la base orthonormée des $u_j$ ; on a
$\|b\|^2 = \sum_j c_j^2 = 1$ et, puisque $\mathbf{S}u_j = \lambda_ju_j$ et $u_j^Tu_k = \delta_{jk}$,
$$b^T\mathbf{S}b = \sum_j\lambda_jc_j^2 \le \lambda_1\sum_jc_j^2 = \lambda_1.$$
L'égalité a lieu pour $c = (1, 0, \dots, 0)$, c'est-à-dire $b = u_1$. $\square$

### Solution

On fait le **changement de variable** $b = \mathbf{W}^{1/2}a$, bijectif car
$\mathbf{W}^{1/2}$ est inversible ($a = \mathbf{W}^{-1/2}b$).

- La contrainte devient $a^T\mathbf{W}a = a^T\mathbf{W}^{1/2}\mathbf{W}^{1/2}a = b^Tb = 1$
  (on a utilisé la symétrie de $\mathbf{W}^{1/2}$).
- L'objectif devient $a^T\mathbf{B}a = b^T\mathbf{W}^{-1/2}\mathbf{B}\mathbf{W}^{-1/2}b = b^T\mathbf{B}^*b$
  avec $\mathbf{B}^* = \mathbf{W}^{-1/2}\mathbf{B}\mathbf{W}^{-1/2}$, symétrique.

Le problème est donc $\max_{\|b\|=1} b^T\mathbf{B}^*b$, un problème aux valeurs propres
**standard** : par le lemme, la solution est le premier vecteur propre $v_1^*$ de
$\mathbf{B}^*$, la valeur optimale est sa plus grande valeur propre $\lambda_1$, et
$$a_1 = \mathbf{W}^{-1/2}v_1^*.$$

**Variantes équivalentes.**

1. Avec la factorisation de Cholesky $\mathbf{W} = \mathbf{L}\mathbf{L}^T$ ($\mathbf{L}$
   triangulaire inférieure), on pose $b = \mathbf{L}^Ta$ et l'on diagonalise
   $\mathbf{L}^{-1}\mathbf{B}\mathbf{L}^{-T}$ - moins coûteux qu'une racine carrée.
2. En multipliant $\mathbf{B}^*v^* = \lambda v^*$ à gauche par $\mathbf{W}^{-1/2}$ et en
   posant $a = \mathbf{W}^{-1/2}v^*$, on obtient $\mathbf{W}^{-1}\mathbf{B}a = \lambda a$ :
   $a_1$ est aussi vecteur propre de $\mathbf{W}^{-1}\mathbf{B}$, mais cette matrice n'est
   **pas symétrique**, d'où l'intérêt de la forme symétrique $\mathbf{B}^*$ pour le calcul.
3. Les directions suivantes ($a_\ell$ maximisant $a^T\mathbf{B}a$ sous $a^T\mathbf{W}a = 1$ et
   $a^T\mathbf{W}a_m = 0$ pour $m<\ell$) sont $a_\ell = \mathbf{W}^{-1/2}v_\ell^*$, car la
   $\mathbf{W}$-orthogonalité des $a$ équivaut à l'orthogonalité usuelle des $b$.

### Vérification numérique

On tire au hasard $\mathbf{B}$ (de rang 3) et $\mathbf{W}$, puis on compare les quatre
calculs : racine carrée, Cholesky, $\mathbf{W}^{-1}\mathbf{B}$ et le solveur généralisé de
scipy (`linalg.eigh(B, W)`).

In [2]:
rng = np.random.RandomState(0)
p = 5
A_ = rng.randn(p, p)
W = A_ @ A_.T + np.eye(p)
G = rng.randn(p, 3)
B = G @ G.T

dW, UW = np.linalg.eigh(W)
W_m12 = UW @ np.diag(dW ** -0.5) @ UW.T
l1, v1 = np.linalg.eigh(W_m12 @ B @ W_m12)
a_racine = W_m12 @ v1[:, -1]

Lw = np.linalg.cholesky(W)
Li = np.linalg.inv(Lw)
l2, v2 = np.linalg.eigh(Li @ B @ Li.T)
a_chol = Li.T @ v2[:, -1]

l3, v3 = np.linalg.eig(np.linalg.solve(W, B))
a_nonsym = np.real(v3[:, np.argmax(np.real(l3))])
a_nonsym /= np.sqrt(a_nonsym @ W @ a_nonsym)

l4, v4 = linalg.eigh(B, W)
a_scipy = v4[:, -1]

for nom, a in [("racine", a_racine), ("Cholesky", a_chol), ("W^-1 B", a_nonsym), ("scipy", a_scipy)]:
    a = a * np.sign(a[0])
    print("%-9s a^T W a = %.6f   a^T B a = %.6f   a = %s" % (nom, a @ W @ a, a @ B @ a, np.round(a, 4)))

racine    a^T W a = 1.000000   a^T B a = 4.349415   a = [ 0.1501  0.1909 -0.6837  0.4358  0.1067]
Cholesky  a^T W a = 1.000000   a^T B a = 4.349415   a = [ 0.1501  0.1909 -0.6837  0.4358  0.1067]
W^-1 B    a^T W a = 1.000000   a^T B a = 4.349415   a = [ 0.1501  0.1909 -0.6837  0.4358  0.1067]
scipy     a^T W a = 1.000000   a^T B a = 4.349415   a = [ 0.1501  0.1909 -0.6837  0.4358  0.1067]
